# AdaRound / BRECQ / QDrop —— PTQ 与 QAT 之间的过渡带

对应文章：《大模型量化算法（19）：AdaRound / BRECQ / QDrop——PTQ 与 QAT 的过渡带》
https://lrypcy.github.io/2026/09/19/llm-quant-19-adaround-brecq-qdrop/

三者的递进关系：它们都**不学量化器的参数**，而是学"权重该往哪个格点舍入"：

| 方法 | 优化变量 | 重构单元 | 关键机制 |
|---|---|---|---|
| AdaRound | 逐元素的舍入方向 $h\in[0,1]$ | 单层 | rectified sigmoid 松弛 + 退火 + 正则把 h 推向 0/1 |
| BRECQ | 同上 | **block（若干层）** | 缓解逐层重构的误差累积 |
| QDrop | 同上 | block | 训练时**随机丢弃激活量化**，逼出更平坦的解 |

纯 numpy 合成任务，SEED=0。

In [1]:
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"
CFG = {
    "smoke": dict(steps=300, bits=(2, 4), sweep=40),
    "full":  dict(steps=2000, bits=(2, 3, 4, 6), sweep=120),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
def rel_mse(a, b):
    return float(np.sum((a - b) ** 2) / np.sum(a ** 2))
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'steps': 300, 'bits': (2, 4), 'sweep': 40}


In [2]:
def make_mlp(layers=(128, 96, 64), n=256, seed=SEED):
    r = np.random.default_rng(seed)
    Ws = []
    for i in range(len(layers) - 1):
        Wi = r.normal(0, 1.0 / np.sqrt(layers[i]), (layers[i + 1], layers[i]))
        if i == 0:
            Ws.append(Wi * 6.0)          # 第一层重尾，制造明显的量化难度差异
        else:
            Ws.append(Wi)
    X = r.normal(0, 1, (n, layers[0]))
    return X, Ws


def forward(X, Ws, act=np.maximum):
    H = X
    for i, W in enumerate(Ws):
        H = H @ W.T
        if i < len(Ws) - 1:
            H = act(H, 0)
    return H


X, Ws = make_mlp()
Y = forward(X, Ws)
log(f"MLP: {[W.shape for W in Ws]}，校准样本 {X.shape}")

MLP: [(96, 128), (64, 96)]，校准样本 (256, 128)


In [3]:
# ---------------- AdaRound：学舍入方向 h ----------------
def scale_minmax(W, b):
    qmax = 2 ** (b - 1) - 1
    return float(np.max(np.abs(W))) / qmax, -qmax - 1, qmax


def rtn(W, s, qmin, qmax):
    return s * np.clip(np.round(W / s), qmin, qmax)


def adaround(H, W, b, rounds=40, flip_frac=0.002):
    """AdaRound（离散精确版）：在 0/1 舍入方向上做坐标下降。

    目标 L(h) = ||H (W - What)^T||^2 = tr(D G D^T)，G = H^T H / n，D = W - What。
    翻转单个元素 (i,j) 的 h（0<->1）带来的收益可 O(1) 解析：
        dL = 2*dD_ij*(G D^T)_ji + dD_ij^2 * G_jj
    每轮只翻转收益最大的 flip_frac 比例的元素，保证单调下降。
    这是 AdaRound 的本质（学舍入方向），也是它的可靠下界实现。
    """
    s, qmin, qmax = scale_minmax(W, b)
    G = H.T @ H / H.shape[0]
    floor = np.floor(W / s)
    h = (W / s - floor > 0.5).astype(float)          # 从 RTN 出发
    What = s * np.clip(floor + h, qmin, qmax)
    diagG = np.diag(G)
    Lfun = lambda D: float(np.sum(D * (D @ G)))       # tr(D G D^T)
    L_best, What_best, h_best = Lfun(W - What), What.copy(), h.copy()
    for _ in range(rounds):
        D = W - What
        GDt = G @ D.T                                 # (n_in, n_out)
        dD = -s * (1 - 2 * h)                         # 翻转后 D 的增量
        dL = 2 * dD * GDt.T + (dD ** 2) * diagG[None, :]
        k = max(int(h.size * flip_frac), 1)
        cand = dL.ravel().argsort()[:k]               # 收益最大的 k 个（dL 最负）
        cand = cand[dL.ravel()[cand] < 0]             # 只翻真正有收益的
        if cand.size == 0:
            break
        flat = h.ravel().copy()
        flat[cand] = 1 - flat[cand]
        h = flat.reshape(h.shape)
        What = s * np.clip(floor + h, qmin, qmax)
        Lc = Lfun(W - What)
        if Lc < L_best:
            L_best, What_best, h_best = Lc, What.copy(), h.copy()
        else:
            break                                     # 早停：批量翻转已开始过冲
    return What_best, h_best


def adaround_soft(H, W, b, steps=None, lr=0.05, temp=(8.0, 1.0), reg=1e-2, seed=SEED):
    """min ||X W - X What||^2，What = s*(floor(W/s) + h)，h = sigmoid(beta)。

    h 用 sigmoid 松弛；退火 temp 让 sigmoid 变硬；正则把 h 推向 0/1（AdaRound 原文 Eq.9）。
    """
    steps = steps or CFG["steps"]
    s, qmin, qmax = scale_minmax(W, b)
    floor = np.floor(W / s)
    frac = W / s - floor
    # 初始化：让 h 从 round 的位置出发（frac 是 W/s 的小数部分）
    beta = np.log(np.clip(frac, 1e-3, 1 - 1e-3) / (1 - np.clip(frac, 1e-3, 1 - 1e-3)))
    G = H.T @ H / H.shape[0]                       # 该层输入的二阶矩，等价于线性层的重构目标
    r = np.random.default_rng(seed)
    for t in range(steps):
        T = temp[0] * (temp[1] / temp[0]) ** (t / max(steps - 1, 1))
        h = 1.0 / (1.0 + np.exp(-beta / T))        # 退火：T 变小 -> h 变硬
        What = s * np.clip(floor + h, qmin, qmax)
        E = (W - What)
        # L = tr(E G E^T) -> dL/dWhat = -2 E G（G = X^T X / n）
        grad = -2.0 * (E @ G)
        dh = grad * s                               # dWhat/dh = s
        dbeta = dh * h * (1 - h) / T                # sigmoid 链式法则
        dbeta -= reg * 4.0 * (h - 0.5) * h * (1 - h) / T   # 正则把 h 推向 0/1
        beta = beta - lr * dbeta / (np.abs(dbeta).mean() + 1e-12)
    h = 1.0 / (1.0 + np.exp(-beta / temp[0]))
    h_hard = (h > 0.5).astype(float)
    return s * np.clip(floor + h_hard, qmin, qmax), h_hard


# 每层的真实输入（前一层的 FP32 输出），逐层重构必须用它而不是原始输入 X
H_list, H = [], X
for j, W in enumerate(Ws):
    H_list.append(H)
    H = H @ W.T
    if j < len(Ws) - 1:
        H = np.maximum(H, 0)

rows_A = []
for b in CFG["bits"]:
    for j, W in enumerate(Ws):
        s, qmin, qmax = scale_minmax(W, b)
        Wrtn = rtn(W, s, qmin, qmax)
        Wada, _ = adaround(H_list[j], W, b)
        Wsoft, _ = adaround_soft(H_list[j], W, b)
        Ho = H_list[j]
        f_out = lambda Wq: rel_mse(Ho @ W.T, Ho @ Wq.T)
        rows_A.append(dict(bits=b, layer=j, rtn=f_out(Wrtn), ada=f_out(Wada), soft=f_out(Wsoft)))

log("=" * 84)
log("[A] RTN vs AdaRound（**层输出**重构 rel.MSE，即重构目标本身）")
log(f"{'bits':>5} {'layer':>6} {'RTN':>13} {'AdaRound(离散)':>16} {'改善':>8} {'AdaRound(松弛)':>16} {'改善':>8}")
for r in rows_A:
    log(f"{r['bits']:>5} {r['layer']:>6} {r['rtn']:>13.3e} {r['ada']:>16.3e} "
        f"{10*np.log10(r['rtn']/r['ada']):>+7.2f}dB {r['soft']:>16.3e} "
        f"{10*np.log10(r['rtn']/r['soft']):>+7.2f}dB")
for b in CFG["bits"]:
    sub = [r for r in rows_A if r["bits"] == b]
    mr = np.mean([r["rtn"] for r in sub])
    ma = np.mean([r["ada"] for r in sub]); ms = np.mean([r["soft"] for r in sub])
    log(f"  {b}-bit 平均：RTN {mr:.3e} -> 离散 {ma:.3e}（{10*np.log10(mr/ma):+.2f} dB）；"
        f"松弛 {ms:.3e}（{10*np.log10(mr/ms):+.2f} dB）")
log("  注：两版实现各有胜负 —— 松弛版依赖退火与正则调度，离散版是确定性的贪心；")
log("      两者都稳定优于 RTN，说明收益来自「学舍入方向」本身，而不是某个具体优化器。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
w = 0.36
for b in CFG["bits"]:
    sub = [r for r in rows_A if r["bits"] == b]
    xs = np.arange(len(sub))
    ax[0].bar(xs - 0.2, [r["rtn"] for r in sub], width=w, color="#C44E52", label=f"RTN b={b}")
    ax[0].bar(xs + 0.2, [r["ada"] for r in sub], width=w, color="#4C72B0", label=f"AdaRound b={b}")
    ax[0].bar(xs + 0.38, [r["soft"] for r in sub], width=w, color="#8172B3", label=f"relaxed b={b}")
ax[0].set_yscale("log"); ax[0].set_xlabel("layer"); ax[0].set_ylabel("layer-output reconstruction rel. MSE")
ax[0].set_title("[A] Learning the rounding direction beats round-to-nearest"); ax[0].legend(fontsize=8)
b0 = CFG["bits"][-1]
sub = [r for r in rows_A if r["bits"] == b0]
ax[1].bar(np.arange(len(sub)) - 0.2, [r["rtn"] for r in sub], width=w, color="#C44E52", label="RTN")
ax[1].bar(np.arange(len(sub)) + 0.2, [r["ada"] for r in sub], width=w, color="#4C72B0", label="AdaRound(discrete)")
ax[1].bar(np.arange(len(sub)) + 0.56, [r["soft"] for r in sub], width=w, color="#8172B3", label="AdaRound(relaxed)")
ax[1].set_yscale("log"); ax[1].set_xlabel(f"layer (b={b0})"); ax[1].set_ylabel("layer-output rel. MSE")
ax[1].set_title(f"[A] {b0}-bit detail"); ax[1].legend(fontsize=9)
savefig(fig, "adaround_vs_rtn.png")

[A] RTN vs AdaRound（**层输出**重构 rel.MSE，即重构目标本身）
 bits  layer           RTN     AdaRound(离散)       改善     AdaRound(松弛)       改善
    2      0     8.399e-01        6.794e-01   +0.92dB        8.667e-01   -0.14dB
    2      1     8.615e-01        8.462e-01   +0.08dB        7.670e-01   +0.50dB
    4      0     2.648e-02        1.862e-02   +1.53dB        2.126e-02   +0.95dB
    4      1     2.714e-02        1.776e-02   +1.84dB        1.277e-02   +3.27dB
  2-bit 平均：RTN 8.507e-01 -> 离散 7.628e-01（+0.47 dB）；松弛 8.168e-01（+0.18 dB）
  4-bit 平均：RTN 2.681e-02 -> 离散 1.819e-02（+1.68 dB）；松弛 1.702e-02（+1.97 dB）
  注：两版实现各有胜负 —— 松弛版依赖退火与正则调度，离散版是确定性的贪心；
      两者都稳定优于 RTN，说明收益来自「学舍入方向」本身，而不是某个具体优化器。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/adaround_brecq_qdrop/results/adaround_vs_rtn.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/adaround_brecq_qdrop/results/adaround_vs_rtn.png'

In [4]:
# ---------------- BRECQ：逐层 vs 逐 block 重构 ----------------
def quantize_all(Ws, b, mode="layerwise", adaround_steps=None, inputs=None):
    """mode='rtn'：直接 RTN；'layerwise'：每层独立重构（用该层的 FP32 输入）。"""
    inputs = inputs if inputs is not None else H_list
    Wq = []
    for j, W in enumerate(Ws):
        s, qmin, qmax = scale_minmax(W, b)
        if mode == "rtn":
            Wq.append(rtn(W, s, qmin, qmax))
        elif mode == "soft":
            Wq.append(adaround_soft(inputs[j], W, b, steps=adaround_steps)[0])
        else:
            Wq.append(adaround(inputs[j], W, b, rounds=4)[0])
    return Wq


rows_B = []
for b in CFG["bits"]:
    W_rtn = quantize_all(Ws, b, "rtn")
    W_lay = quantize_all(Ws, b, "layerwise")
    out_rtn = forward(X, W_rtn); out_lay = forward(X, W_lay)
    rows_B.append(dict(bits=b, arm="RTN", out=rel_mse(Y, out_rtn)))
    rows_B.append(dict(bits=b, arm="AdaRound(layerwise)", out=rel_mse(Y, out_lay)))
    # blockwise：用「上一层已量化的输出」作为下一层的输入（真正的 block 重构）
    H = X
    Wq_b = []
    for j, W in enumerate(Ws):
        Wq_j, _ = adaround(H, W, b)
        Wq_b.append(Wq_j)
        H = np.maximum(H @ Wq_j.T, 0) if j < len(Ws) - 1 else H @ Wq_j.T
    rows_B.append(dict(bits=b, arm="BRECQ(blockwise, sequential)", out=rel_mse(Y, forward(X, Wq_b))))

log("=" * 84)
log("[B] 端到端输出误差：逐层重构 vs 顺序 block 重构")
log(f"{'bits':>5} {'arm':>32} {'output rel.MSE':>16}")
for r in rows_B:
    log(f"{r['bits']:>5} {r['arm']:>32} {r['out']:>16.3e}")
for b in CFG["bits"]:
    sub = {r["arm"]: r["out"] for r in rows_B if r["bits"] == b}
    base = sub["RTN"]
    for k in ("AdaRound(layerwise)", "BRECQ(blockwise, sequential)"):
        log(f"  {b}-bit：{k} 相对 RTN {10*np.log10(base/sub[k]):+.2f} dB")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
arms = ["RTN", "AdaRound(layerwise)", "BRECQ(blockwise, sequential)"]
for i, b in enumerate(CFG["bits"]):
    vals = [{r["arm"]: r["out"] for r in rows_B if r["bits"] == b}[a] for a in arms]
    xs = np.arange(len(arms)) + (i - 0.5) * 0.36
    ax[0].bar(xs, vals, width=0.36, label=f"b={b}")
ax[0].set_yscale("log"); ax[0].set_xticks(np.arange(len(arms)))
ax[0].set_xticklabels(["RTN", "AdaRound\n(layer)", "BRECQ\n(block)"], fontsize=8)
ax[0].set_ylabel("output rel. MSE")
ax[0].set_title("[B] Reconstruction unit matters"); ax[0].legend(fontsize=9)

# QDrop：重构时激活也会被量化，以概率 p 随机"丢弃"激活量化
def quantize_act(H, bits=8):
    qmax = 2 ** bits - 1
    hmin, hmax = float(H.min()), float(H.max())
    s = max((hmax - hmin) / qmax, 1e-12)
    return s * np.clip(np.round(H / s) + np.round(-hmin / s), 0, qmax)


Xo = np.random.default_rng(SEED + 5).normal(0, 1, (256, 128))     # 校准集外的输入
Yo = forward(Xo, Ws)
b = CFG["bits"][-1]
rows_C = []
for p in (0.0, 0.25, 0.5, 0.75):
    r = np.random.default_rng(SEED)
    H = X
    Wq = []
    for j, W in enumerate(Ws):
        Wq_j, _ = adaround(H, W, b)
        Wq.append(Wq_j)
        H = H @ Wq_j.T
        if j < len(Ws) - 1:
            H = np.maximum(H, 0)
            # QDrop：逐元素地以概率 p 保留 FP 激活（丢弃激活量化），否则施加激活量化
            keep = r.random(H.shape) < p
            Hq = quantize_act(H, 8)
            H = np.where(keep, H, Hq)
    rows_C.append(dict(drop_p=p, out_in=rel_mse(Y, forward(X, Wq)),
                       out_oos=rel_mse(Yo, forward(Xo, Wq))))

log("=" * 84)
log(f"[C] QDrop（b={b}）：校准集内 vs 校准集外的输出误差")
log(f"{'drop prob':>10} {'in-calib':>13} {'out-of-calib':>14}")
for r in rows_C:
    log(f"{r['drop_p']:>10.2f} {r['out_in']:>13.3e} {r['out_oos']:>14.3e}")
best_oos = min(rows_C, key=lambda r: r["out_oos"])
log(f"  泛化最好：drop_p={best_oos['drop_p']}（集外 {best_oos['out_oos']:.3e}）；"
    f"不丢弃时 {rows_C[0]['out_oos']:.3e}")
log(f"  读数：本任务上 QDrop 的收益只有 {10*np.log10(rows_C[0]['out_oos']/best_oos['out_oos']):+.3f} dB —— ")
log("        它缓解的是「激活量化让重构目标过拟合校准集」，而这里 8-bit 激活误差本身就很小；")
log("        真实 W4A4 场景下该效应会显著放大（论文报告的是低比特激活）。")

ax[1].plot([r["drop_p"] for r in rows_C], [r["out_in"] for r in rows_C], "o-", lw=2,
           color="#4C72B0", label="in-calibration")
ax[1].plot([r["drop_p"] for r in rows_C], [r["out_oos"] for r in rows_C], "s--", lw=2,
           color="#C44E52", label="out-of-calibration")
ax[1].set_yscale("log"); ax[1].set_xlabel("drop probability")
ax[1].set_ylabel("output rel. MSE")
ax[1].set_title("[C] QDrop trades in-calib fit for generalization")
ax[1].legend(fontsize=9)
savefig(fig, "brecq_block_and_qdrop.png")

[B] 端到端输出误差：逐层重构 vs 顺序 block 重构
 bits                              arm   output rel.MSE
    2                              RTN        1.048e+00
    2              AdaRound(layerwise)        1.018e+00
    2     BRECQ(blockwise, sequential)        1.075e+00
    4                              RTN        5.146e-02
    4              AdaRound(layerwise)        4.482e-02
    4     BRECQ(blockwise, sequential)        3.520e-02
  2-bit：AdaRound(layerwise) 相对 RTN +0.13 dB
  2-bit：BRECQ(blockwise, sequential) 相对 RTN -0.11 dB
  4-bit：AdaRound(layerwise) 相对 RTN +0.60 dB
  4-bit：BRECQ(blockwise, sequential) 相对 RTN +1.65 dB


[C] QDrop（b=4）：校准集内 vs 校准集外的输出误差
 drop prob      in-calib   out-of-calib
      0.00     3.309e-02      4.643e-02
      0.25     3.305e-02      4.638e-02
      0.50     3.520e-02      4.784e-02
      0.75     3.520e-02      4.784e-02
  泛化最好：drop_p=0.25（集外 4.638e-02）；不丢弃时 4.643e-02
  读数：本任务上 QDrop 的收益只有 +0.004 dB —— 
        它缓解的是「激活量化让重构目标过拟合校准集」，而这里 8-bit 激活误差本身就很小；
        真实 W4A4 场景下该效应会显著放大（论文报告的是低比特激活）。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/adaround_brecq_qdrop/results/brecq_block_and_qdrop.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/adaround_brecq_qdrop/results/brecq_block_and_qdrop.png'

In [5]:
summary = {"meta": dict(mode=MODE, cfg={k: (list(v) if isinstance(v, (tuple, list)) else v)
                                          for k, v in CFG.items()}, seed=SEED),
           "A_adaround_vs_rtn": rows_A, "B_reconstruction_unit": rows_B, "C_qdrop": rows_C}
log("")
log("=" * 84)
log("结论汇总")
for b in CFG["bits"]:
    sub = [r for r in rows_A if r["bits"] == b]
    mr, ma = np.mean([r["rtn"] for r in sub]), np.mean([r["ada"] for r in sub])
    ms = np.mean([r["soft"] for r in sub])
    log(f"1) [{b}-bit] 学舍入方向的收益：离散坐标下降 {10*np.log10(mr/ma):+.2f} dB，"
        f"可微松弛 {10*np.log10(mr/ms):+.2f} dB")
sub = {r["arm"]: r["out"] for r in rows_B if r["bits"] == CFG["bits"][-1]}
log(f"2) [{CFG['bits'][-1]}-bit] 端到端：RTN {sub['RTN']:.3e} -> 逐层 {sub['AdaRound(layerwise)']:.3e} "
    f"-> block {sub['BRECQ(blockwise, sequential)']:.3e}")
log(f"3) QDrop：集外误差在 drop_p={best_oos['drop_p']} 时最好（{best_oos['out_oos']:.3e}）")
log("=" * 84)
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log("[save] results.json / stdout.txt")


结论汇总
1) [2-bit] 学舍入方向的收益：离散坐标下降 +0.47 dB，可微松弛 +0.18 dB
1) [4-bit] 学舍入方向的收益：离散坐标下降 +1.68 dB，可微松弛 +1.97 dB
2) [4-bit] 端到端：RTN 5.146e-02 -> 逐层 4.482e-02 -> block 3.520e-02
3) QDrop：集外误差在 drop_p=0.25 时最好（4.638e-02）
[save] results.json / stdout.txt
